# 04 · LLM follow-up agent, evaluated with simulated people

For a night the detector flags, the agent shows the person what was observed and asks a few
short questions about the day before. It stores what they report as validated JSON: factor,
timing, how sure they were, and their own words. It never says what caused the night and gives
no diagnosis. The protocol is decision 49 in `docs/decisions.md`, committed before the test run.

- **What code fixes, and what the LLM writes:**
  - Code writes the opening: the observed facts, rendered from the anomaly summary, then the
    disclaimer and one open question.
  - The LLM writes up to 4 follow-up questions. Code blocks any that suggests a cause or a
    diagnosis.
  - Code writes the closing.
- **Extraction:** after the chat, the model lists what the person talked about (happened, did not
  happen, or routine), and code turns that into the `FollowUpResult`.
- **Agents:**
  - `gemma3:4b`, the default;
  - `medgemma:4b`, the same base with medical tuning;
  - `llama3.2:3b`;
  - a fixed checklist with the same opening and 4 fixed neutral questions. Its extraction is
    done by `gemma3:4b`.
- **Simulated people:**
  - Played by `qwen3.5:9b`, which also judges.
  - Each person has 0–3 hidden causes (about 20% outside the checklist) and two ordinary events.
  - There are three styles: forthcoming, reticent and vague.
  - Causes are drawn independently of the anomaly. No real causes are known, so the agent can't
    be rewarded for guessing from the data.
- **Test set:** 60 personas on 20 cases, 5 real flagged nights and 15 injected ones. Every agent
  talks to the same 60, so comparisons are paired. Prompts were tuned only on separate dev
  personas and cases (other subjects), then frozen.

**What this shows and what it doesn't.** These are LLM-played people with a fixed card. They are
more consistent than real people, so the numbers are an upper bound for real use. The notebook
only reads saved runs.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sleepwatch.agent.cases import AgentCase
from sleepwatch.agent.evaluate import AgentEvalConfig
from sleepwatch.agent.judge import Judgement, judged_questions
from sleepwatch.agent.personas import Persona
from sleepwatch.agent.schema import FollowUpRecord
from sleepwatch.config import get_settings

RUN_NAME = "agent_test"  # the single test run (decision 49)
settings = get_settings()
index = pd.read_csv(settings.results_dir / "agent_index.csv")
done = index[(index["name"] == RUN_NAME) & ~index["partial"] & index["complete"]]
run_dir = settings.results_dir / RUN_NAME / done.iloc[-1]["run"]
info = json.loads((run_dir / "run.json").read_text())
cfg = AgentEvalConfig.from_yaml(run_dir / "config.yaml")
print(
    f"{RUN_NAME} run {run_dir.name} at {info['git']['commit'][:8]} "
    f"(dirty: {info['git']['dirty']}), prompts {info['prompt_version']}, "
    f"Ollama {info['ollama']['version']}"
)
AGENTS = [a.name for a in cfg.agents]
REF = cfg.reference
personas = {
    p["persona_id"]: Persona(**p) for p in json.loads((run_dir / "personas.json").read_text())
}
cases = {c["case_id"]: AgentCase(**c) for c in json.loads((run_dir / "cases.json").read_text())}
records, judged = {}, {}
for line in (run_dir / "conversations.jsonl").read_text().splitlines():
    row = json.loads(line)
    records[(row["agent"], row["persona_id"])] = FollowUpRecord.model_validate(row["record"])
for line in (run_dir / "judgements.jsonl").read_text().splitlines():
    j = Judgement.model_validate_json(line)
    judged[(j.agent, j.persona_id)] = j
metrics = json.loads((run_dir / "metrics.json").read_text())
table = pd.read_parquet(run_dir / "counts.parquet")
print(
    f"{len(personas)} personas x {len(AGENTS)} agents = {len(records)} conversations; "
    f"{sum(c.split == cfg.split for c in cases.values())} cases"
)

plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": "#a9a8a3",
        "axes.labelcolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e6e5e1",
        "grid.linewidth": 0.6,
        "lines.linewidth": 1.5,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "font.size": 9,
        "legend.frameon": False,
    }
)
COLORS = dict(zip(AGENTS, ["#2a78d6", "#eb6834", "#1f9e74", "#8a8984"], strict=False))


def ci(entry, digits=2):
    return f"{entry['value']:.{digits}f} [{entry['lo']:.{digits}f}, {entry['hi']:.{digits}f}]"


def show(agent, pid, width=150):
    """One conversation with the judge's labels, the extraction and the hidden card."""
    rec, j, p = records[(agent, pid)], judged[(agent, pid)], personas[pid]
    print(f"--- {agent} · {pid} · case {rec.case_id}")
    print("hidden:", [f"{c.factor} ({c.detail}, {c.timing})" for c in p.causes] or "nothing")
    labels = {i: q.label for i, q in zip(judged_questions(rec), j.questions, strict=False)}
    for i, t in enumerate(rec.turns):
        who = "Person" if t.role == "person" else "Agent"
        tag = f" [{labels[i]}]" if i in labels else (f" [{t.kind}]" if t.role == "agent" else "")
        text = t.text if i else t.text.split("\n\n")[-1]  # the opening's question only
        print(f"  {who}{tag}: {text[:width]}")
        for draft, hits in zip(t.drafts, t.guard, strict=False):
            print(f"      blocked draft ({'; '.join(hits)}): {draft[:width]}")
    for k, f in enumerate(rec.result.factors):
        s = j.factors[k] if j.factors else None
        ok = s is not None and s.supported and j.quote_found[k]
        print(
            f'  -> {f.factor} | {f.timing} | {f.confidence} | "{f.quote[:80]}" | '
            f"{'supported' if ok else 'NOT supported'}"
        )
    print(f"  denied: {rec.result.denied}; nothing unusual: {rec.result.nothing_unusual}")

## 1 · What the person sees

The opening is code, not LLM: the facts from the anomaly summary, the disclaimer and one open
question. Here it is for a real flagged night.

In [ ]:
split_cases = [c for c in cases.values() if c.split == cfg.split]
real = next((c for c in split_cases if c.kind == "real"), split_cases[0])
first = next(r for r in records.values() if r.case_id == real.case_id)
print(f"case {real.case_id} ({real.kind})\n")
print(first.turns[0].text)

Then the same simulated person, talking to each agent. Each agent question carries the judge's
label, and any draft the guard blocked is shown. Below the conversation are the extracted
factors, each marked supported or not (its quote must appear in the person's words, and the
judge must agree).

In [ ]:
pairs = [p for p in personas.values() if len(p.causes) == 2]
pid = next(
    (
        p.persona_id
        for p in pairs
        if p.style == "reticent" and any(c.factor == "other" for c in p.causes)
    ),
    pairs[0].persona_id,
)
for agent in AGENTS:
    show(agent, pid)
    print()

## 2 · Primary metrics

Pooled over conversations, with 95% CIs from 2000 bootstrap resamples of personas.
- **Cause recall:** the share of hidden causes captured. A cause is captured by a supported
  factor that the judge matches to it or that has the same category, one factor per cause.
- **Precision:** the share of extracted factors that capture a hidden cause.
- **Hallucinated factors:** extracted factors the person's words don't support.
- **Leading questions:** the share of follow-up questions the judge calls leading.

The differences are paired by persona, against the default `gemma3:4b`.

In [ ]:
PRIMARY = {
    "cause_recall": "cause recall",
    "cause_precision": "precision",
    "hallucinated_factor_rate": "hallucinated factors",
    "leading_question_rate": "leading questions",
    "turns_to_elicit_median": "median questions to elicit",
}
display(
    pd.DataFrame(
        {
            agent: {label: ci(metrics["agents"][agent][k]) for k, label in PRIMARY.items()}
            for agent in AGENTS
        }
    )
)
print(f"differences against {REF} (paired by persona):")
display(
    pd.DataFrame(
        {
            agent: {
                label: (f"{d[k]['value']:+.2f} [{d[k]['lo']:+.2f}, {d[k]['hi']:+.2f}]")
                for k, label in PRIMARY.items()
            }
            for agent, d in metrics["differences"].items()
        }
    )
)

fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), sharey=True)
for ax, key in zip(axes, list(PRIMARY)[:4], strict=True):
    for y, agent in enumerate(AGENTS):
        e = metrics["agents"][agent][key]
        ax.errorbar(
            e["value"],
            y,
            xerr=[[e["value"] - e["lo"]], [e["hi"] - e["value"]]],
            fmt="o",
            color=COLORS[agent],
            capsize=3,
        )
    ax.set(title=PRIMARY[key], xlim=(-0.02, 1.02))
axes[0].set_yticks(range(len(AGENTS)), AGENTS)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

## 3 · Asking vs extracting

Missing a cause can happen in two places:
- **Asking:** the person never mentions the cause, which the disclosure rate measures.
- **Extracting:** the person mentions it but the extraction misses it, which extraction recall
  on disclosed causes measures.

The curve shows how many hidden causes had come up after each question. Question 1 is the
templated open question, the same for every agent.

In [ ]:
display(
    pd.DataFrame(
        {
            agent: {
                "disclosed (asking)": ci(metrics["agents"][agent]["disclosure_rate"]),
                "extracted when disclosed": ci(
                    metrics["agents"][agent]["extraction_recall_disclosed"]
                ),
                "cause recall": ci(metrics["agents"][agent]["cause_recall"]),
                "strict recall (same category)": ci(metrics["agents"][agent]["strict_recall"]),
                "category right (captured causes)": ci(
                    metrics["agents"][agent]["category_accuracy"]
                ),
            }
            for agent in AGENTS
        }
    )
)
fig, ax = plt.subplots(figsize=(6, 3))
# distinct markers and dashes, so agents with identical curves stay visible
styles = [("o", "-", 9), ("s", "--", 7), ("D", "-.", 5), ("^", ":", 4)]
for agent, (marker, dash, size) in zip(AGENTS, styles, strict=False):
    by_turn = metrics["agents"][agent]["disclosed_by_turn"]
    ax.plot(
        range(1, len(by_turn) + 1),
        by_turn,
        marker=marker,
        markersize=size,
        linestyle=dash,
        color=COLORS[agent],
        label=agent,
    )
ax.set(
    xlabel="questions asked",
    ylabel="share of hidden causes disclosed",
    ylim=(0, 1),
    xticks=range(1, 6),
    title="When do causes come up?",
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4 · By kind of person and kind of cause

Point values per stratum: 20 personas per style and 15 per number of causes. "Other" causes are
outside the checklist, such as a nap, new medication or noise. Only an open question finds
those.

In [ ]:
def strata_frame(by):
    rows = []
    for agent in AGENTS:
        for level, values in metrics["strata"][by][agent].items():
            rows.append({"agent": agent, by: level, **values})
    return pd.DataFrame(rows).set_index(["agent", by]).round(2)


display(strata_frame("style"))
display(strata_frame("n_causes"))
display(
    pd.DataFrame(
        {
            agent: {
                "recall, checklist causes": ci(metrics["agents"][agent]["recall_checklist"]),
                "recall, other causes": ci(metrics["agents"][agent]["recall_other"]),
                "timing agrees (matched causes)": ci(metrics["agents"][agent]["timing_agreement"]),
                "certainty agrees (matched causes)": ci(
                    metrics["agents"][agent]["confidence_agreement"]
                ),
                "no-cause personas given a factor": ci(
                    metrics["agents"][agent]["no_cause_factor_rate"]
                ),
            }
            for agent in AGENTS
        }
    )
)

## 5 · Safety: leading questions, causes and diagnoses

What the judge flags in the questions the person saw, and what the guard caught before that. A
blocked draft is regenerated once and then replaced by a fixed neutral question (*fallback*).

In [ ]:
display(
    pd.DataFrame(
        {
            agent: {
                "leading (seen)": ci(metrics["agents"][agent]["leading_question_rate"]),
                "leading or blocked (before guard)": ci(
                    metrics["agents"][agent]["leading_before_guard_rate"]
                ),
                "causal statements (seen)": ci(metrics["agents"][agent]["causal_statement_rate"]),
                "diagnosis statements (seen)": ci(
                    metrics["agents"][agent]["diagnosis_statement_rate"]
                ),
                "questions with a blocked draft": ci(
                    metrics["agents"][agent]["guard_trigger_rate"]
                ),
            }
            for agent in AGENTS
        }
    )
)
fallbacks = pd.Series(
    {
        a: sum(t.kind == "fallback" for (b, _), r in records.items() if b == a for t in r.turns)
        for a in AGENTS
    },
    name="fallback questions",
)
display(fallbacks.to_frame())
print("Questions the judge called leading (up to 4 per agent):")
for agent in AGENTS:
    shown = 0
    for (a, pid), rec in records.items():
        if a != agent or shown == 4:
            continue
        for i, q in zip(judged_questions(rec), judged[(a, pid)].questions, strict=False):
            if q.label == "leading" and shown < 4:
                print(f"  {agent:12} {rec.turns[i].text[:140]}")
                shown += 1
print("\nDrafts the guard blocked (up to 6):")
blocked = [
    (a, d, h)
    for (a, _), rec in records.items()
    for t in rec.turns
    for d, h in zip(t.drafts, t.guard, strict=False)
]
for a, d, h in blocked[:6]:
    print(f"  {a:12} [{'; '.join(h)}] {d[:120]}")

## 6 · Extraction errors

These are factors the person's words don't support, and causes that came up in the conversation
but weren't extracted.

In [ ]:
unsupported, missed = [], []
for (agent, pid), rec in records.items():
    j, p = judged[(agent, pid)], personas[pid]
    if not j.factors and rec.result.factors:
        continue  # judge call failed; counted in the metrics as excluded
    for k, f in enumerate(rec.result.factors):
        if not (j.factors[k].supported and j.quote_found[k]):
            unsupported.append(
                {
                    "agent": agent,
                    "persona": pid,
                    "factor": f.factor,
                    "quote": f.quote[:70],
                    "quote found": j.quote_found[k],
                    "judge: supported": j.factors[k].supported,
                }
            )
    reported = {f.factor for f in rec.result.factors}
    for c, d in zip(p.causes, j.causes, strict=False):
        if d.first_answer is not None and c.factor not in reported:
            missed.append({"agent": agent, "persona": pid, "cause": f"{c.factor}: {c.detail}"})
unsupported = pd.DataFrame(unsupported)
missed = pd.DataFrame(missed)
print(f"unsupported factors: {len(unsupported)}; disclosed causes not extracted: {len(missed)}")
if len(unsupported):
    display(unsupported.groupby("agent").size().rename("unsupported").to_frame())
    display(unsupported.head(12))
if len(missed):
    display(missed.groupby("agent").size().rename("missed").to_frame())
    display(missed.head(12))
display(
    pd.DataFrame(
        {
            agent: {
                "valid on the first try": ci(metrics["agents"][agent]["schema_valid_first_try"]),
                "repaired by code": ci(metrics["agents"][agent]["extraction_repaired"]),
                "failed": ci(metrics["agents"][agent]["extraction_failed"]),
            }
            for agent in AGENTS
        }
    )
)

## 7 · How faithful are the simulated people?

A simulated person can stray from its card in two ways:
- **Drift:** it claims something that isn't on its card. The extraction can then be right about
  the conversation but wrong about the card, and precision suffers for a reason that isn't the
  agent's.
- **Denial:** it says a cause on its card didn't happen. Recall then suffers, and the extraction
  sees a contradiction.

The judge checks this answer by answer. To show how much of it that check finds, code planted a
claim (and, for cards with causes, a denial) in the last answer of some conversations; the
detection rates are below. Under 80%, the drift and denial rates are lower bounds (decision
49).

In [ ]:
display(
    pd.DataFrame(
        {
            "drift": {a: ci(metrics["agents"][a]["simulator_drift_rate"]) for a in AGENTS},
            "denial": {a: ci(metrics["agents"][a]["simulator_denial_rate"]) for a in AGENTS},
        }
    )
)
probes = metrics["judge_probes"]
for kind, p in probes.items():
    print(f"planted {kind}s detected: {p['detected']}/{p['n']}  ({p['errors']} judge failures)")
extra = [(a, pid, claim) for (a, pid), j in judged.items() for claim in j.extra_claims]
print(f"\n{len(extra)} claims not on the card, e.g.:")
for a, pid, claim in extra[:8]:
    print(f"  {a:12} {pid:22} {claim[:110]}")
denied = [
    (a, pid, personas[pid].causes[c - 1].detail)
    for (a, pid), j in judged.items()
    for c in j.denied_causes
]
print(f"\n{len(denied)} card causes denied, e.g.:")
for a, pid, detail in denied[:8]:
    print(f"  {a:12} {pid:22} {detail[:110]}")
errors = sum(bool(j.errors) for j in judged.values())
print(f"\njudge calls that failed after retries: {errors} conversations")

## 8 · How far to trust the judge

**Consistency.** The first question is the same templated question for every agent, and the
simulated person's sampling is seeded. So each person's first answer is word-for-word identical
across agents. The judge sees that answer four times, inside four different conversations. If
it is consistent, it marks the same hidden causes as disclosed in it each time.

In [ ]:
first, at_first = {}, {}
for (agent, pid), rec in records.items():
    first.setdefault(pid, set()).add(rec.turns[1].text)
    j = judged[(agent, pid)]
    if not any(e.startswith("disclosures") for e in j.errors):
        marked = tuple(sorted(c.cause for c in j.causes if c.first_answer == 1))
        at_first.setdefault(pid, {})[agent] = marked
identical = sum(len(texts) == 1 for texts in first.values())
judged_twice = {pid: v for pid, v in at_first.items() if len(v) > 1}
consistent = sum(len(set(v.values())) == 1 for v in judged_twice.values())
print(f"first answers identical across agents: {identical} of {len(first)} personas")
print(
    f"judge marks the same causes in that answer for every agent: {consistent} of "
    f"{len(judged_twice)} personas"
)
for pid, v in judged_twice.items():
    if len(set(v.values())) > 1:
        print(f"  {pid}: {v}")

**Against human labels.** You labelled a blinded sample of agent questions
and extracted factors with the same rubric. Decision 49 accepts the judge if Cohen's κ ≥ 0.6.
Below that, the human-labelled rates take precedence.

In [ ]:
agreement_file = run_dir / "labels" / "agreement.json"
if agreement_file.exists():
    agree = json.loads(agreement_file.read_text())
    print(
        f"{agree['questions']} questions, {agree['factors']} factors; "
        f"kappa leading {agree['kappa_leading']:.2f} (raw agreement "
        f"{agree['agreement_leading']:.2f}), kappa supported {agree['kappa_supported']:.2f} "
        f"(raw {agree['agreement_supported']:.2f}); 3-class question kappa "
        f"{agree['kappa_label_3class']:.2f}"
    )
    rows = {}
    for agent in AGENTS:
        h = agree["human_leading_rate"].get(agent)
        u = agree["human_unsupported_rate"].get(agent)
        rows[agent] = {
            "leading, human sample": f"{h['value']:.2f} [{h['lo']:.2f}, {h['hi']:.2f}] (n={h['n']})"
            if h
            else "-",
            "leading, judge (all)": ci(metrics["agents"][agent]["leading_question_rate"]),
            "unsupported, human sample": f"{u['value']:.2f} [{u['lo']:.2f}, {u['hi']:.2f}] "
            f"(n={u['n']})"
            if u
            else "-",
            "unsupported, judge (all)": ci(metrics["agents"][agent]["hallucinated_factor_rate"]),
        }
    display(pd.DataFrame(rows))
else:
    print("No human labels yet: run `sleepwatch agent label-sample` and fill in the sheet.")

## 9 · Speed on this machine

These are median seconds per step, measured on an M3 with 16 GB that was sharing memory with
other work. They're relevant to Phase 5 (a chat must answer in seconds), but they depend on the
machine and its load, so they say nothing about the methods.

In [ ]:
rows = {}
for agent in AGENTS:
    recs = [r for (a, _), r in records.items() if a == agent]
    asked = [t.seconds for r in recs for t in r.turns if t.kind in ("question", "fallback")]
    answered = [t.seconds for r in recs for t in r.turns if t.kind == "answer"]
    rows[agent] = {
        "agent question": np.median(asked) if asked and any(asked) else np.nan,
        "extraction": np.median([r.extraction_seconds for r in recs]),
        "simulated person": np.median(answered),
    }
display(pd.DataFrame(rows).round(1))

## 10 · Findings

*To be written after the test run.*